
# OpenAI Director Agent 기반 동영상 생성 실습
## Agent가 창작 Workflow 전체를 지휘하는 방법

이 Notebook은 앞선 **ChatGPT → Copilot → Application Agent → Image Agent** 강의의 다음 단계입니다.

핵심 구조:

**Goal → Storyboard → Base Frame → Cinematic Prompt → Renderer → Evaluation → Human Approval → Final MP4**

> **중요:** OpenAI Videos API와 Sora 2 계열은 2026-09-24 API에서 종료되었습니다. 따라서 본 실습은 **OpenAI Director Agent + GPT Image + OpenAI TTS + FFmpeg Renderer**를 결합한 교육용 Agentic Video Production Pipeline입니다.  
> Renderer는 Adapter로 분리되어 있으므로, 향후 별도의 Video Generation API로 교체할 수 있습니다.

### 실행 모드
- `RUN_OPENAI_MEDIA = False` : 비용 없는 Mock 이미지/무음 오디오로 전체 Workflow 검증
- `RUN_OPENAI_MEDIA = True` : OpenAI Image + TTS를 실제 호출

먼저 Mock mode로 끝까지 실행한 뒤 실제 Media Tool을 켜는 것을 권장합니다.



## LAB 0 — 환경 설정
Colab에서 다음 Cell을 실행합니다. `OPENAI_API_KEY`는 Colab **Secrets**에 저장하는 것을 권장합니다.


In [ ]:

!pip -q install -U openai pydantic pillow
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:

from __future__ import annotations

import base64
import json
import os
import shutil
import subprocess
import textwrap
from pathlib import Path
from typing import Literal

from IPython.display import Video, display, Image as IPyImage
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel, Field
from openai import OpenAI

# --- Colab Secret 우선 사용 ---
try:
    from google.colab import userdata
    if not os.getenv("OPENAI_API_KEY"):
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
except Exception:
    pass

# ===== Models =====
TEXT_MODEL = os.getenv("OPENAI_TEXT_MODEL", "gpt-5.5")
IMAGE_MODEL = os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-2.5-flare")
TTS_MODEL = os.getenv("OPENAI_TTS_MODEL", "gpt-4o-mini-tts")

# ===== Cost / execution controls =====
RUN_OPENAI_MEDIA = False    # 실제 Image/TTS 호출 시 True
RUN_VISION_EVAL = False     # OpenAI Vision 평가 시 True
RUN_AUTONOMOUS_AGENT = False
INTERACTIVE_APPROVAL = False

# ===== Working directory =====
ROOT = Path("/content/director_agent_lab")
FRAMES = ROOT / "frames"
AUDIO = ROOT / "audio"
CLIPS = ROOT / "clips"
OUTPUT = ROOT / "output"
for p in [ROOT, FRAMES, AUDIO, CLIPS, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
client = OpenAI() if HAS_KEY else None

print("TEXT_MODEL:", TEXT_MODEL)
print("IMAGE_MODEL:", IMAGE_MODEL)
print("TTS_MODEL:", TTS_MODEL)
print("API key available:", HAS_KEY)
print("RUN_OPENAI_MEDIA:", RUN_OPENAI_MEDIA)



## LAB 1 — User Goal → Storyboard
Agent는 곧바로 렌더러를 호출하지 않습니다. 먼저 전체 영상의 **목적, Scene, 시간 배분, 시각적 일관성**을 설계합니다.


In [ ]:

MISSION = """
기업 임직원 대상 'AI Agent와 Enterprise AI' 강의의 20초 오프닝 영상을 완성해 주세요.

조건:
- 16:9 교육용 시네마틱 스타일
- 정확히 3개 Scene
- 전문적이고 절제된 분위기
- 마지막 Scene은 'From Answer to Action'의 의미가 전달되어야 함
- 장면 간 동일한 톤과 색감 유지
- 한국어 내레이션 포함
""".strip()

print(MISSION)


In [ ]:

class Scene(BaseModel):
    scene_id: int
    duration_sec: int = Field(ge=3, le=10)
    purpose: str
    subject: str
    action: str
    scene: str
    camera: str
    camera_movement: Literal["static", "slow_zoom_in", "slow_zoom_out", "pan_left", "pan_right"]
    lighting: str
    style: str
    narration: str
    transition: Literal["cut", "crossfade", "fade"]

class Storyboard(BaseModel):
    title: str
    audience: str
    visual_consistency: str
    scenes: list[Scene]


def mock_storyboard() -> Storyboard:
    return Storyboard(
        title="From Answer to Action",
        audience="Corporate employees learning AI Agents",
        visual_consistency=(
            "Modern enterprise classroom, dark navy and violet accents, "
            "soft cinematic lighting, realistic educational documentary style."
        ),
        scenes=[
            Scene(
                scene_id=1, duration_sec=6,
                purpose="Establish the problem: AI only answers questions.",
                subject="A corporate professional using a conversational AI on a laptop",
                action="The employee reads an AI answer and pauses, thinking about the next task",
                scene="Modern training room with a large presentation screen in the background",
                camera="Eye-level medium-wide shot",
                camera_movement="slow_zoom_in",
                lighting="Soft cool office light with subtle violet screen glow",
                style="Photorealistic professional educational documentary",
                narration="생성형 AI는 이제 답변을 넘어 실제 업무 실행으로 이동하고 있습니다.",
                transition="crossfade",
            ),
            Scene(
                scene_id=2, duration_sec=7,
                purpose="Show an Agent coordinating tools and workflows.",
                subject="An instructor explaining an AI Agent workflow beside a digital screen",
                action="The instructor gestures as connected tool icons and workflow nodes appear on the screen",
                scene="Corporate classroom with clean digital visualization of files, tools and applications",
                camera="Eye-level medium shot",
                camera_movement="pan_right",
                lighting="Balanced neutral lighting with blue and violet UI accents",
                style="Realistic enterprise technology documentary, clean and restrained",
                narration="Agent는 목표를 이해하고, 필요한 도구를 선택하며, 여러 단계를 연결합니다.",
                transition="crossfade",
            ),
            Scene(
                scene_id=3, duration_sec=7,
                purpose="Conclude with outcome-oriented enterprise AI.",
                subject="A completed presentation and connected enterprise workflow visual on a large screen",
                action="The screen transitions from a chat bubble to connected workflow nodes and a completed artifact",
                scene="Minimal dark navy presentation environment with elegant violet and teal highlights",
                camera="Centered hero shot",
                camera_movement="slow_zoom_out",
                lighting="Cinematic rim lighting with controlled high contrast",
                style="Premium corporate keynote visual, realistic and minimal",
                narration="이제 질문하는 AI에서, 결과를 완성하는 AI로 전환합니다.",
                transition="fade",
            ),
        ],
    )


def plan_storyboard(mission: str) -> Storyboard:
    if not HAS_KEY:
        print("[Mock] API key가 없어 예제 Storyboard를 사용합니다.")
        return mock_storyboard()

    response = client.responses.parse(
        model=TEXT_MODEL,
        input=[
            {
                "role": "system",
                "content": (
                    "You are a professional Director Agent for short corporate educational videos. "
                    "Convert the user's goal into exactly 3 scenes. Total duration should be around 20 seconds. "
                    "Keep narration concise enough to fit each scene. Maintain strong visual continuity."
                ),
            },
            {"role": "user", "content": mission},
        ],
        text_format=Storyboard,
    )
    board = response.output_parsed
    if board is None:
        raise RuntimeError("Storyboard Structured Output을 파싱하지 못했습니다.")
    return board

storyboard = plan_storyboard(MISSION)
print(storyboard.model_dump_json(indent=2))

(ROOT / "storyboard.json").write_text(
    storyboard.model_dump_json(indent=2), encoding="utf-8"
)



### Storyboard 확인
Structured Output은 Creative Idea를 **실행 가능한 Tool Contract**로 바꿉니다.


In [ ]:

for s in storyboard.scenes:
    print(f"SCENE {s.scene_id} | {s.duration_sec}s | {s.camera_movement}")
    print("Purpose:", s.purpose)
    print("Narration:", s.narration)
    print("-" * 80)



## LAB 2 — Scene → Cinematic Prompt
Image Prompt보다 Video Production Prompt는 **시간, 카메라 이동, 전환**을 포함해야 합니다.


In [ ]:

def scene_to_image_prompt(scene: Scene, consistency: str) -> str:
    return f"""
Create a 16:9 cinematic keyframe for a corporate educational video.

VISUAL CONSISTENCY:
{consistency}

SUBJECT:
{scene.subject}

ACTION / MOMENT:
{scene.action}

SCENE:
{scene.scene}

CAMERA:
{scene.camera}

LIGHTING:
{scene.lighting}

STYLE:
{scene.style}

Requirements:
- Photorealistic, professional, restrained.
- Clean composition with clear subject separation.
- Do not render captions, subtitles, logos, watermarks, or readable UI text.
- Leave reasonable negative space for later titles if appropriate.
""".strip()


def scene_to_cinematic_prompt(scene: Scene) -> str:
    return f"""
Scene purpose: {scene.purpose}
Subject: {scene.subject}
Action: {scene.action}
Environment: {scene.scene}
Camera framing: {scene.camera}
Camera movement: {scene.camera_movement}
Lighting: {scene.lighting}
Style: {scene.style}
Duration: {scene.duration_sec} seconds
Narration: {scene.narration}
Transition: {scene.transition}
""".strip()

for scene in storyboard.scenes:
    print(f"
=== Scene {scene.scene_id} Cinematic Prompt ===")
    print(scene_to_cinematic_prompt(scene))



## LAB 3 — Base Frame 생성
`RUN_OPENAI_MEDIA=False`일 때는 Mock Frame을 생성합니다. True로 바꾸면 GPT Image를 호출합니다.


In [ ]:

def make_mock_frame(scene: Scene, out_path: Path) -> Path:
    img = Image.new("RGB", (1536, 1024), color=(18, 28, 48))
    draw = ImageDraw.Draw(img)
    # Safe ASCII text for Colab default fonts
    title = f"SCENE {scene.scene_id}"
    lines = [
        title,
        scene.purpose[:72],
        f"Camera: {scene.camera}",
        f"Movement: {scene.camera_movement}",
        "MOCK FRAME - turn RUN_OPENAI_MEDIA=True for GPT Image",
    ]
    y = 260
    for i, line in enumerate(lines):
        draw.text((120, y), line, fill=(245, 247, 250) if i < 2 else (160, 180, 215))
        y += 90 if i == 0 else 65
    # visual accents
    draw.rectangle((90, 175, 1446, 185), fill=(124, 58, 237))
    draw.rectangle((90, 790, 1446, 800), fill=(15, 118, 110))
    img.save(out_path)
    return out_path


def generate_stage_image(scene: Scene) -> Path:
    out_path = FRAMES / f"scene_{scene.scene_id:02d}.png"
    if out_path.exists():
        return out_path

    if RUN_OPENAI_MEDIA:
        if not HAS_KEY:
            raise RuntimeError("OPENAI_API_KEY가 필요합니다.")
        result = client.images.generate(
            model=IMAGE_MODEL,
            prompt=scene_to_image_prompt(scene, storyboard.visual_consistency),
            size="1536x1024",
            quality="medium",
        )
        image_bytes = base64.b64decode(result.data[0].b64_json)
        out_path.write_bytes(image_bytes)
    else:
        make_mock_frame(scene, out_path)

    return out_path

frame_paths = [generate_stage_image(s) for s in storyboard.scenes]
frame_paths


In [ ]:

for p in frame_paths:
    display(IPyImage(filename=str(p), width=600))



## LAB 4 — Keyframe Evaluation
선택 실습입니다. OpenAI Vision으로 생성된 Base Frame이 Scene 목적과 맞는지 평가합니다.


In [ ]:

def evaluate_keyframe(scene: Scene, image_path: Path) -> dict:
    if not (RUN_VISION_EVAL and HAS_KEY):
        return {
            "status": "SKIPPED",
            "reason": "RUN_VISION_EVAL=False",
            "scene_id": scene.scene_id,
        }

    b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    prompt = f"""
Evaluate whether this keyframe is suitable for the planned corporate educational video scene.
Scene purpose: {scene.purpose}
Subject: {scene.subject}
Action: {scene.action}
Scene: {scene.scene}
Camera: {scene.camera}
Lighting: {scene.lighting}
Style: {scene.style}

Return a concise assessment with PASS or REVISE and 2-3 reasons.
""".strip()
    response = client.responses.create(
        model=TEXT_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"data:image/png;base64,{b64}", "detail": "auto"},
            ],
        }],
    )
    return {"status": "DONE", "assessment": response.output_text}

for s, p in zip(storyboard.scenes, frame_paths):
    print(s.scene_id, evaluate_keyframe(s, p))



## LAB 5 — OpenAI TTS Narration
실제 TTS 사용 시 AI-generated voice임을 학습자/시청자에게 명확히 고지해야 합니다.


In [ ]:

def make_silence(duration_sec: int, out_path: Path) -> Path:
    cmd = [
        "ffmpeg", "-y", "-f", "lavfi",
        "-i", "anullsrc=r=44100:cl=stereo",
        "-t", str(duration_sec),
        "-q:a", "9", "-acodec", "libmp3lame", str(out_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_path


def synthesize_narration(scene: Scene) -> Path:
    out_path = AUDIO / f"scene_{scene.scene_id:02d}.mp3"
    if out_path.exists():
        return out_path

    if RUN_OPENAI_MEDIA:
        if not HAS_KEY:
            raise RuntimeError("OPENAI_API_KEY가 필요합니다.")
        with client.audio.speech.with_streaming_response.create(
            model=TTS_MODEL,
            voice="marin",
            input=scene.narration,
            instructions="Speak in clear, calm, professional Korean lecture narration. Avoid dramatic delivery.",
        ) as response:
            response.stream_to_file(out_path)
    else:
        make_silence(scene.duration_sec, out_path)

    return out_path

audio_paths = [synthesize_narration(s) for s in storyboard.scenes]
audio_paths



## LAB 6 — Video Renderer Tool
OpenAI의 네이티브 Video API 대신 FFmpeg를 **교체 가능한 Renderer Adapter**로 사용합니다.

이 단계의 교육 포인트는 고급 영상효과가 아니라 다음입니다.

**Director Agent → Renderer Tool Contract → MP4 Artifact**


In [ ]:

def _zoom_expr(movement: str) -> tuple[str, str, str]:
    # Returns z, x, y expressions for FFmpeg zoompan.
    if movement == "slow_zoom_out":
        return (
            "if(eq(on,0),1.08,max(1.0,zoom-0.00045))",
            "iw/2-(iw/zoom/2)",
            "ih/2-(ih/zoom/2)",
        )
    if movement == "pan_left":
        return (
            "1.06",
            "max(0,(iw-iw/zoom)*(1-on/(duration*30)))",
            "ih/2-(ih/zoom/2)",
        )
    if movement == "pan_right":
        return (
            "1.06",
            "min(iw-iw/zoom,(iw-iw/zoom)*(on/(duration*30)))",
            "ih/2-(ih/zoom/2)",
        )
    if movement == "static":
        return ("1.0", "0", "0")
    # default slow_zoom_in
    return (
        "min(1.08,zoom+0.00045)",
        "iw/2-(iw/zoom/2)",
        "ih/2-(ih/zoom/2)",
    )


def render_scene_video(scene: Scene, frame_path: Path, audio_path: Path) -> Path:
    video_only = CLIPS / f"scene_{scene.scene_id:02d}_video.mp4"
    final_clip = CLIPS / f"scene_{scene.scene_id:02d}.mp4"
    duration = scene.duration_sec
    z, x, y = _zoom_expr(scene.camera_movement)
    # substitute duration variable used by pan expressions
    x = x.replace("duration", str(duration))
    y = y.replace("duration", str(duration))

    vf = (
        f"scale=1920:1080:force_original_aspect_ratio=increase,"
        f"crop=1920:1080,"
        f"zoompan=z='{z}':x='{x}':y='{y}':d={duration*30}:s=1920x1080:fps=30,"
        f"format=yuv420p"
    )

    cmd1 = [
        "ffmpeg", "-y", "-loop", "1", "-i", str(frame_path),
        "-vf", vf, "-t", str(duration), "-r", "30",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", str(video_only),
    ]
    subprocess.run(cmd1, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    cmd2 = [
        "ffmpeg", "-y", "-i", str(video_only), "-i", str(audio_path),
        "-filter_complex", "[1:a]apad[a]",
        "-map", "0:v", "-map", "[a]",
        "-c:v", "copy", "-c:a", "aac", "-b:a", "160k",
        "-t", str(duration), "-shortest", str(final_clip),
    ]
    subprocess.run(cmd2, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return final_clip

clip_paths = [
    render_scene_video(scene, frame, audio)
    for scene, frame, audio in zip(storyboard.scenes, frame_paths, audio_paths)
]
clip_paths



## LAB 7 — Final Cut
Scene MP4를 연결하고 `manifest.json`을 함께 저장합니다.


In [ ]:

def assemble_final_video(clips: list[Path], output_path: Path) -> Path:
    concat_file = ROOT / "concat.txt"
    concat_file.write_text(
        "
".join([f"file '{p.resolve()}'" for p in clips]), encoding="utf-8"
    )
    cmd = [
        "ffmpeg", "-y", "-f", "concat", "-safe", "0",
        "-i", str(concat_file),
        "-c:v", "libx264", "-c:a", "aac", "-pix_fmt", "yuv420p",
        str(output_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path

final_video = assemble_final_video(clip_paths, OUTPUT / "final_video.mp4")

manifest = {
    "mission": MISSION,
    "models": {
        "text": TEXT_MODEL,
        "image": IMAGE_MODEL if RUN_OPENAI_MEDIA else "mock",
        "tts": TTS_MODEL if RUN_OPENAI_MEDIA else "silent-mock",
        "video_renderer": "ffmpeg-keyframe-adapter",
    },
    "storyboard": storyboard.model_dump(),
    "artifacts": {
        "frames": [str(p) for p in frame_paths],
        "audio": [str(p) for p in audio_paths],
        "clips": [str(p) for p in clip_paths],
        "final_video": str(final_video),
    },
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("Final:", final_video)
print("Manifest:", OUTPUT / "manifest.json")


In [ ]:

display(Video(str(final_video), embed=True, width=900))



# PART 2 — Director Agent가 Tool을 직접 Orchestration하기
지금까지는 사람이 Cell 순서를 실행했습니다. 이제 **Responses API Function Calling**으로 Director Agent가 Tool 호출 순서를 결정하도록 확장합니다.

Tool set:

- `plan_storyboard`
- `generate_stage_image`
- `synthesize_narration`
- `render_scene_video`
- `evaluate_scene`
- `assemble_final_video`
- `request_human_approval`

> 이 실습은 API Key가 있어야 합니다. `RUN_OPENAI_MEDIA=False`여도 Director Agent의 Tool Calling 자체는 OpenAI Text Model을 호출합니다.


In [ ]:

# Agent runtime state
AGENT_STATE = {
    "mission": None,
    "storyboard": None,
    "frames": {},
    "audio": {},
    "clips": {},
    "final_video": None,
    "approved": False,
}


def get_scene(scene_id: int) -> Scene:
    board = AGENT_STATE.get("storyboard")
    if board is None:
        raise RuntimeError("Storyboard가 없습니다. plan_storyboard를 먼저 호출하세요.")
    for scene in board.scenes:
        if scene.scene_id == scene_id:
            return scene
    raise ValueError(f"Unknown scene_id={scene_id}")


def tool_plan_storyboard(mission: str) -> dict:
    board = plan_storyboard(mission)
    AGENT_STATE["mission"] = mission
    AGENT_STATE["storyboard"] = board
    return {
        "ok": True,
        "title": board.title,
        "scene_ids": [s.scene_id for s in board.scenes],
        "scenes": [s.model_dump() for s in board.scenes],
    }


def tool_generate_stage_image(scene_id: int) -> dict:
    scene = get_scene(scene_id)
    p = generate_stage_image(scene)
    AGENT_STATE["frames"][scene_id] = str(p)
    return {"ok": True, "scene_id": scene_id, "frame_path": str(p)}


def tool_synthesize_narration(scene_id: int) -> dict:
    scene = get_scene(scene_id)
    p = synthesize_narration(scene)
    AGENT_STATE["audio"][scene_id] = str(p)
    return {"ok": True, "scene_id": scene_id, "audio_path": str(p)}


def tool_render_scene_video(scene_id: int) -> dict:
    scene = get_scene(scene_id)
    frame = Path(AGENT_STATE["frames"].get(scene_id, ""))
    audio = Path(AGENT_STATE["audio"].get(scene_id, ""))
    if not frame.exists() or not audio.exists():
        return {"ok": False, "error": "Frame과 Audio가 먼저 필요합니다.", "scene_id": scene_id}
    p = render_scene_video(scene, frame, audio)
    AGENT_STATE["clips"][scene_id] = str(p)
    return {"ok": True, "scene_id": scene_id, "clip_path": str(p)}


def tool_evaluate_scene(scene_id: int) -> dict:
    scene = get_scene(scene_id)
    frame_path = Path(AGENT_STATE["frames"].get(scene_id, ""))
    clip_path = Path(AGENT_STATE["clips"].get(scene_id, ""))
    checks = {
        "frame_exists": frame_path.exists(),
        "clip_exists": clip_path.exists(),
        "duration_target_sec": scene.duration_sec,
        "narration_present": bool(scene.narration.strip()),
        "camera_movement": scene.camera_movement,
    }
    status = "PASS" if all([checks["frame_exists"], checks["clip_exists"], checks["narration_present"]]) else "REVISE"
    return {"ok": True, "scene_id": scene_id, "status": status, "checks": checks}


def tool_assemble_final_video() -> dict:
    board = AGENT_STATE.get("storyboard")
    if board is None:
        return {"ok": False, "error": "Storyboard가 없습니다."}
    ordered = []
    for scene in board.scenes:
        p = Path(AGENT_STATE["clips"].get(scene.scene_id, ""))
        if not p.exists():
            return {"ok": False, "error": f"Scene {scene.scene_id} clip이 없습니다."}
        ordered.append(p)
    p = assemble_final_video(ordered, OUTPUT / "agent_final_video.mp4")
    AGENT_STATE["final_video"] = str(p)
    return {"ok": True, "final_video": str(p)}


def tool_request_human_approval(summary: str) -> dict:
    if INTERACTIVE_APPROVAL:
        answer = input(f"
[HUMAN APPROVAL]
{summary}
Approve? (y/n): ").strip().lower()
        approved = answer in {"y", "yes", "approve", "approved"}
    else:
        approved = True  # classroom demo default
    AGENT_STATE["approved"] = approved
    return {"ok": True, "approved": approved, "mode": "interactive" if INTERACTIVE_APPROVAL else "demo-auto"}

IMPLEMENTATIONS = {
    "plan_storyboard": tool_plan_storyboard,
    "generate_stage_image": tool_generate_stage_image,
    "synthesize_narration": tool_synthesize_narration,
    "render_scene_video": tool_render_scene_video,
    "evaluate_scene": tool_evaluate_scene,
    "assemble_final_video": lambda: tool_assemble_final_video(),
    "request_human_approval": tool_request_human_approval,
}


In [ ]:

TOOLS = [
    {
        "type": "function",
        "name": "plan_storyboard",
        "description": "Create a 3-scene storyboard for the user's video mission and store it in runtime state.",
        "parameters": {
            "type": "object",
            "properties": {"mission": {"type": "string"}},
            "required": ["mission"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "generate_stage_image",
        "description": "Generate or mock the base keyframe image for one planned scene.",
        "parameters": {
            "type": "object",
            "properties": {"scene_id": {"type": "integer"}},
            "required": ["scene_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "synthesize_narration",
        "description": "Generate or mock narration audio for one planned scene.",
        "parameters": {
            "type": "object",
            "properties": {"scene_id": {"type": "integer"}},
            "required": ["scene_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "render_scene_video",
        "description": "Render one scene MP4 from its keyframe and narration. Image and audio must exist first.",
        "parameters": {
            "type": "object",
            "properties": {"scene_id": {"type": "integer"}},
            "required": ["scene_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "evaluate_scene",
        "description": "Check whether a rendered scene has the expected artifacts and is ready for final assembly.",
        "parameters": {
            "type": "object",
            "properties": {"scene_id": {"type": "integer"}},
            "required": ["scene_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "assemble_final_video",
        "description": "Concatenate all rendered scene clips into the final MP4. Call only after all scenes pass evaluation.",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
        "strict": True,
    },
    {
        "type": "function",
        "name": "request_human_approval",
        "description": "Request final human approval after the final video artifact exists.",
        "parameters": {
            "type": "object",
            "properties": {"summary": {"type": "string"}},
            "required": ["summary"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


In [ ]:

def run_director_agent(mission: str, max_turns: int = 30) -> str:
    if not HAS_KEY:
        raise RuntimeError("Director Agent Tool Calling에는 OPENAI_API_KEY가 필요합니다.")

    # Reset runtime state
    AGENT_STATE.update({
        "mission": None, "storyboard": None,
        "frames": {}, "audio": {}, "clips": {},
        "final_video": None, "approved": False,
    })

    input_items = [{"role": "user", "content": mission}]
    instructions = """
You are the Director Agent for a corporate educational video studio.
Your job is to COMPLETE the production workflow, not merely explain it.

Required execution policy:
1. Call plan_storyboard first.
2. For every planned scene, call generate_stage_image and synthesize_narration.
3. Then call render_scene_video for every scene.
4. Call evaluate_scene for every scene. If a tool returns an error, fix prerequisites and retry.
5. Only after all scenes PASS, call assemble_final_video.
6. After the final MP4 exists, call request_human_approval.
7. Finish with a concise summary listing the final artifact path and approval state.

Do not claim a file exists unless a tool returned that path.
""".strip()

    for turn in range(1, max_turns + 1):
        response = client.responses.create(
            model=TEXT_MODEL,
            instructions=instructions,
            input=input_items,
            tools=TOOLS,
            tool_choice="auto",
        )

        # Preserve ALL model output items, including reasoning/tool calls.
        input_items.extend(item.model_dump(exclude_none=True) for item in response.output)
        calls = [item for item in response.output if item.type == "function_call"]

        if not calls:
            return response.output_text or "Director Agent completed without final text."

        for call in calls:
            args = json.loads(call.arguments)
            fn = IMPLEMENTATIONS.get(call.name)
            if fn is None:
                result = {"ok": False, "error": f"Unknown tool: {call.name}"}
            else:
                try:
                    result = fn(**args)
                except Exception as exc:
                    result = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}

            print(f"[Tool] {call.name}({args}) -> {result}")
            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

    raise RuntimeError("Director Agent가 max_turns 안에 종료되지 않았습니다.")



### Autonomous Director Agent 실행
기본값은 `RUN_AUTONOMOUS_AGENT=False`입니다.  
실습 시 True로 바꿔 Tool Call 순서를 관찰하세요.

- `RUN_OPENAI_MEDIA=False` : Director는 OpenAI 모델이지만 media tools는 Mock + FFmpeg
- `RUN_OPENAI_MEDIA=True` : Director + GPT Image + OpenAI TTS + FFmpeg


In [ ]:

if RUN_AUTONOMOUS_AGENT:
    result = run_director_agent(MISSION)
    print("
=== DIRECTOR AGENT FINAL RESPONSE ===")
    print(result)
    if AGENT_STATE.get("final_video"):
        display(Video(AGENT_STATE["final_video"], embed=True, width=900))
else:
    print("RUN_AUTONOMOUS_AGENT=False — True로 변경하면 Agent Tool Loop를 실행합니다.")



# FINAL CHALLENGE
다음 Mission만 Agent에게 주고, **HOW를 세세하게 지시하지 마세요.**

> **“생성형 AI 업무혁신 강의의 시작에 사용할 20초 오프닝 영상을 완성하십시오.”**

평가 기준:
1. Storyboard가 명확한가?
2. 모든 Scene이 동일한 Visual Tone을 유지하는가?
3. Tool Calling 순서가 합리적인가?
4. Artifact 경로가 실제 존재하는가?
5. Evaluation과 Human Approval이 포함되는가?

### 핵심 정리
- **Video Model = Generator**
- **Director Agent = Planner + Orchestrator**
- **Tool = 전문 실행 모듈**
- **Evaluator = Revision Gate**
- **Human = 최종 책임과 승인**

**Goal → Plan → Tools → Evaluate → Approve → Artifact**



## 참고: 실습 결과 다운로드
Colab 왼쪽 Files 패널에서 `/content/director_agent_lab/output/` 폴더의 다음 파일을 다운로드할 수 있습니다.

- `final_video.mp4`
- `manifest.json`
- `agent_final_video.mp4` (Autonomous Agent 실행 시)
